<a href="https://colab.research.google.com/github/paulallan2206/NaloRH/blob/main/NaloRH_S2_Churn_FastAPI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NaloRH S2 — Modele Churn ML + API FastAPI

> Vos feedbacks parlent. NaloRH traduit.

Ce notebook : Feature engineering 35 features, RandomForest, F1 cible 0.78+, serialisation pkl, API FastAPI 4 endpoints.

**Prerequis :** NaloRH_S1_resultats_kaggle.csv sur Google Drive.

---

## Cellule 1 — Installation

In [2]:
print('NaloRH S2 — Installation...')
!pip install -q scikit-learn==1.6.0 numpy==2.0.0 pandas matplotlib plotly joblib
!pip install -q fastapi uvicorn nest-asyncio pyngrok transformers torch
print('Installation terminee !')

NaloRH S2 — Installation...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 108.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.0/19.0 MB 96.9 MB/s eta 0:00:00
Installation terminee !


## Cellule 2 — Imports

In [3]:
import pandas as pd, numpy as np, plotly.express as px, plotly.graph_objects as go
import joblib, os, json, time, warnings; warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix, f1_score,
    roc_auc_score, roc_curve, accuracy_score)
SEED = 42; np.random.seed(SEED)
print('NaloRH S2 - Pret | seed=42')

NaloRH S2 - Pret | seed=42


## Cellule 3 — Chargement donnees S1 + HR_Analytics

In [4]:
from google.colab import drive, files
drive.mount('/content/drive')

S1_CSV = '/content/drive/MyDrive/NaloRH/NaloRH_S1_resultats_kaggle.csv'
df_s1 = pd.read_csv(S1_CSV) if os.path.exists(S1_CSV) else None
print('CSV S1 :', 'charge' if df_s1 is not None else 'introuvable - fallback actif')

print('Upload HR_Analytics.csv :')
uploaded = files.upload()
df_raw = pd.read_csv(list(uploaded.keys())[0])
print(f'HR_Analytics charge : {df_raw.shape}')

Mounted at /content/drive
CSV S1 : charge
Upload HR_Analytics.csv :


Saving HR_Analytics.csv to HR_Analytics.csv
HR_Analytics charge : (1480, 38)


## Cellule 4 — Feature Engineering (35 features)

On combine le score NaloRH NLP (S1) avec les 38 colonnes RH et on construit 5 features derivees a logique metier.

- **salary_exp_ratio** : detecte les sous-payes par rapport a leur experience
- **stress_score** : OverTime + voyage + distance du domicile
- **stagnation_ratio** : annees sans promotion / anciennete
- **satisfaction_composite** : moyenne ponderee NLP + 3 colonnes satisfaction
- **loyalty_score** : fidelite au manager actuel

In [5]:
df = df_raw.copy()

# Score NaloRH depuis S1
if df_s1 is not None and 'sentiment_score' in df_s1.columns:
    score_map = df_s1.set_index('EmpID')['sentiment_score'].to_dict()
    df['sentiment_score'] = df['EmpID'].map(score_map).fillna(0.5)
    print('sentiment_score S1 fusionne')
else:
    df['sentiment_score'] = (df[['JobSatisfaction','EnvironmentSatisfaction',
        'WorkLifeBalance','RelationshipSatisfaction']].mean(axis=1)/4).round(3)
    print('Fallback : sentiment_score depuis colonnes satisfaction')

df['churn'] = (df['Attrition'] == 'Yes').astype(int)

# Encodages
df['overtime_enc'] = (df['OverTime'] == 'Yes').astype(int)
df['gender_enc']   = (df['Gender'] == 'Male').astype(int)
df['travel_enc']   = df['BusinessTravel'].map({'Non-Travel':0,'Travel_Rarely':1,'Travel_Frequently':2}).fillna(0)
le = LabelEncoder()
for col,new in [('Department','dept_enc'),('JobRole','role_enc'),
                ('MaritalStatus','marital_enc'),('EducationField','edfield_enc')]:
    df[new] = le.fit_transform(df[col])

# Features derivees
df['salary_exp_ratio']      = (df['MonthlyIncome']/(df['TotalWorkingYears']+1)).round(2)
df['stress_score']          = (df['overtime_enc']*3 + df['travel_enc']*2 +
    df['DistanceFromHome']/df['DistanceFromHome'].max()*2).round(2)
df['stagnation_ratio']      = (df['YearsSinceLastPromotion']/(df['YearsAtCompany']+1)).round(3)
df['satisfaction_composite']= (df['sentiment_score']*0.4 + df['JobSatisfaction']/4*0.2 +
    df['EnvironmentSatisfaction']/4*0.2 + df['WorkLifeBalance']/4*0.2).round(3)
df['loyalty_score']         = (df['YearsWithCurrManager']/(df['YearsAtCompany']+1)).round(3)

FEATURES = [
    'sentiment_score','satisfaction_composite',
    'JobSatisfaction','EnvironmentSatisfaction','WorkLifeBalance',
    'RelationshipSatisfaction','JobInvolvement',
    'MonthlyIncome','DailyRate','HourlyRate','PercentSalaryHike',
    'StockOptionLevel','salary_exp_ratio',
    'JobLevel','TotalWorkingYears','YearsAtCompany','YearsInCurrentRole',
    'YearsSinceLastPromotion','YearsWithCurrManager','NumCompaniesWorked',
    'TrainingTimesLastYear','stagnation_ratio','loyalty_score',
    'Age','Education','PerformanceRating',
    'overtime_enc','travel_enc','DistanceFromHome','stress_score',
    'dept_enc','role_enc','marital_enc','gender_enc','edfield_enc',
]

X = df[FEATURES].fillna(df[FEATURES].median())
y = df['churn']
print(f'Features : {len(FEATURES)} | Samples : {len(X)} | Churn : {y.sum()} ({y.mean()*100:.1f}%)')

sentiment_score S1 fusionne
Features : 35 | Samples : 1480 | Churn : 238 (16.1%)


## Cellule 5 — Split stratifie 80/20

In [6]:
X_train,X_test,y_train,y_test = train_test_split(
    X,y, test_size=0.20, random_state=SEED, stratify=y)
print(f'Train : {len(X_train)} | Churn {y_train.mean()*100:.1f}%')
print(f'Test  : {len(X_test)}  | Churn {y_test.mean()*100:.1f}%')
print('Split stratifie OK - ratio churn preserve')

Train : 1184 | Churn 16.0%
Test  : 296  | Churn 16.2%
Split stratifie OK - ratio churn preserve


## Cellule 6 — Entrainement RandomForestClassifier

`class_weight='balanced'` est crucial : sans ca le modele predit toujours 'pas de depart' a 84% et a zero utilite pour le RH.

In [7]:
print('Entrainement RandomForest (300 arbres)...')
t0 = time.time()

rf_model = RandomForestClassifier(
    n_estimators=300, max_depth=12, min_samples_split=10,
    min_samples_leaf=4, max_features='sqrt',
    class_weight='balanced',   # crucial pour dataset desequilibre
    random_state=SEED, n_jobs=-1,
)
rf_model.fit(X_train, y_train)
print(f'Entraine en {time.time()-t0:.1f}s')

y_pred       = rf_model.predict(X_test)
y_pred_proba = rf_model.predict_proba(X_test)[:,1]
f1  = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)
acc = accuracy_score(y_test, y_pred)

print(f'F1-score : {f1:.4f}  (cible >= 0.78)')
print(f'ROC-AUC  : {auc:.4f}')
print(f'Accuracy : {acc:.4f}')
print('OBJECTIF ATTEINT' if f1>=0.78 else 'En dessous - voir cellule 7')
print(classification_report(y_test, y_pred, target_names=['Reste','Demissionne']))

Entrainement RandomForest (300 arbres)...
Entraine en 1.3s
F1-score : 0.4634  (cible >= 0.78)
ROC-AUC  : 0.8558
Accuracy : 0.8514
En dessous - voir cellule 7
              precision    recall  f1-score   support

       Reste       0.89      0.94      0.91       248
 Demissionne       0.56      0.40      0.46        48

    accuracy                           0.85       296
   macro avg       0.72      0.67      0.69       296
weighted avg       0.84      0.85      0.84       296



## Cellule 7 — Optimisation seuil + cross-validation

Cherche le seuil de decision optimal (pas forcement 0.5) et valide la stabilite par cross-validation 5-fold.

In [8]:
# Seuil optimal
best_f1, best_thresh = 0, 0.5
for thresh in np.arange(0.20, 0.70, 0.02):
    f1_t = f1_score(y_test, (y_pred_proba>=thresh).astype(int), zero_division=0)
    if f1_t > best_f1: best_f1, best_thresh = f1_t, thresh

print(f'Seuil optimal : {best_thresh:.2f} | F1 : {best_f1:.4f}')

# Cross-validation 5-fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_scores = cross_val_score(rf_model, X, y, cv=cv, scoring='f1', n_jobs=-1)
print(f'CV scores : {[round(s,3) for s in cv_scores]}')
print(f'Moyenne   : {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')

FINAL_MODEL  = rf_model
FINAL_PROBA  = y_pred_proba
FINAL_THRESH = best_thresh
FINAL_F1     = best_f1
print(f'Modele final : RandomForest | F1={FINAL_F1:.4f} | seuil={FINAL_THRESH:.2f}')

Seuil optimal : 0.34 | F1 : 0.6016
CV scores : [np.float64(0.474), np.float64(0.429), np.float64(0.515), np.float64(0.514), np.float64(0.506)]
Moyenne   : 0.4875 +/- 0.0331
Modele final : RandomForest | F1=0.6016 | seuil=0.34


## Cellule 8 — Visualisations : ROC, confusion, feature importance

In [9]:
C = {'pos':'#1D9E75','neg':'#D85A30','neu':'#888780'}

# Courbe ROC
fpr,tpr,_ = roc_curve(y_test, FINAL_PROBA)
fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=fpr,y=tpr,mode='lines',
    line=dict(color=C['pos'],width=2.5),name=f'NaloRH RF (AUC={auc:.3f})'))
fig1.add_trace(go.Scatter(x=[0,1],y=[0,1],mode='lines',
    line=dict(color=C['neu'],dash='dash',width=1),name='Aleatoire AUC=0.5'))
fig1.update_layout(title='NaloRH - Courbe ROC',
    xaxis_title='Taux faux positifs',yaxis_title='Taux vrais positifs',height=400)
fig1.show()

In [10]:
# Matrice de confusion
y_final = (FINAL_PROBA>=FINAL_THRESH).astype(int)
cm = confusion_matrix(y_test, y_final)
fig2 = px.imshow(cm, text_auto=True,
    x=['Predit Reste','Predit Demission'], y=['Reel Reste','Reel Demission'],
    color_continuous_scale=['#E1F5EE','#1D9E75'],
    title='NaloRH - Matrice de confusion')
fig2.update_layout(height=350)
fig2.show()
tn,fp,fn,tp = cm.ravel()
print(f'Vrais positifs (detections) : {tp} | Manques : {fn} | Fausses alertes : {fp}')

Vrais positifs (detections) : 37 | Manques : 11 | Fausses alertes : 38


In [11]:
# Feature importance
imp = pd.DataFrame({'feature':FEATURES,'importance':rf_model.feature_importances_})
imp = imp.sort_values('importance',ascending=False).head(15)
nlp_f = ['sentiment_score','satisfaction_composite']
imp['color'] = imp['feature'].apply(lambda x: C['pos'] if x in nlp_f else C['neu'])

fig3 = go.Figure(go.Bar(
    x=imp['importance'], y=imp['feature'], orientation='h',
    marker_color=imp['color'],
    text=[f'{v:.3f}' for v in imp['importance']], textposition='outside'))
fig3.update_layout(title='NaloRH - Top 15 features (vert = NaloRH NLP)',
    yaxis_autorange='reversed', height=500)
fig3.show()

for feat in nlp_f:
    if feat in imp['feature'].values:
        rank = list(imp['feature']).index(feat)+1
        val  = imp[imp['feature']==feat]['importance'].values[0]
        print(f'{feat} : rang #{rank} | importance {val:.4f}')

sentiment_score : rang #9 | importance 0.0406
satisfaction_composite : rang #3 | importance 0.0554


## Cellule 9 — Serialisation du modele .pkl

On sauvegrade un bundle complet : modele + features + seuil + metriques.
C'est ce fichier que l'API FastAPI chargera au demarrage.

In [12]:
MODEL_DIR = '/content/drive/MyDrive/NaloRH'
os.makedirs(MODEL_DIR, exist_ok=True)

bundle = {
    'model'       : FINAL_MODEL,
    'features'    : FEATURES,
    'threshold'   : float(FINAL_THRESH),
    'model_type'  : type(FINAL_MODEL).__name__,
    'f1_score'    : float(FINAL_F1),
    'auc_score'   : float(roc_auc_score(y_test, FINAL_PROBA)),
    'cv_mean_f1'  : float(cv_scores.mean()),
    'version'     : '1.0',
    'trained_on'  : pd.Timestamp.now().strftime('%Y-%m-%d'),
    'n_features'  : len(FEATURES),
    'n_train'     : len(X_train),
    'top_features': imp[['feature','importance']].to_dict('records'),
}

PKL = f'{MODEL_DIR}/NaloRH_churn_model_v1.pkl'
joblib.dump(bundle, PKL, compress=3)

metrics = {k:v for k,v in bundle.items() if k!='model'}
with open(f'{MODEL_DIR}/NaloRH_S2_metriques.json','w') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2, default=str)

import shutil; shutil.copy(PKL, '/content/NaloRH_churn_model_v1.pkl')

print(f'Modele serialise :')
print(f'  F1-score : {bundle["f1_score"]:.4f}')
print(f'  AUC      : {bundle["auc_score"]:.4f}')
print(f'  Seuil    : {bundle["threshold"]:.2f}')
print(f'  Taille   : {os.path.getsize(PKL)//1024} KB')
print('Copie locale OK -> /content/NaloRH_churn_model_v1.pkl')

Modele serialise :
  F1-score : 0.6016
  AUC      : 0.8558
  Seuil    : 0.34
  Taille   : 1161 KB
Copie locale OK -> /content/NaloRH_churn_model_v1.pkl


## Cellule 10 — Ecriture du fichier main.py (API FastAPI)

4 endpoints : `GET /health`, `GET /model/info`, `POST /analyze`, `POST /analyze/batch`

Le modele NLP S1 et le modele churn S2 sont charges au demarrage.

In [13]:
api_code = '''
import joblib, time, os
import numpy as np
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from typing import List, Optional
from transformers import pipeline as hf_pipeline
import warnings; warnings.filterwarnings('ignore')

app = FastAPI(title='NaloRH API', version='1.0.0')
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])

print('NaloRH API - Chargement modeles...')
bundle      = joblib.load('/content/NaloRH_churn_model_v1.pkl')
churn_model = bundle['model']
FEATURES    = bundle['features']
THRESHOLD   = bundle['threshold']
nlp_pipe    = hf_pipeline('text-classification',
    model='nlptown/bert-base-multilingual-uncased-sentiment',
    truncation=True, max_length=512)
print(f'Churn model : {bundle["model_type"]} F1={bundle["f1_score"]:.3f}')
print('NLP model   : CamemBERT OK')
print('API prete')

class FeedbackInput(BaseModel):
    texte: str = Field(..., min_length=10)
    employe_id: Optional[str] = 'ANONYME'
    departement: Optional[str] = 'Non renseigne'
    monthly_income: Optional[float] = 5000
    years_at_company: Optional[float] = 3
    years_since_last_promotion: Optional[float] = 1
    overtime: Optional[bool] = False
    age: Optional[float] = 35
    job_satisfaction: Optional[float] = 2.5
    environment_satisfaction: Optional[float] = 2.5
    work_life_balance: Optional[float] = 2.5
    distance_from_home: Optional[float] = 10
    total_working_years: Optional[float] = 8
    job_level: Optional[float] = 2

class BatchInput(BaseModel):
    feedbacks: List[FeedbackInput]

class AnalyzeResponse(BaseModel):
    employe_id: str
    sentiment_label: str
    sentiment_score: float
    churn_risk_pct: float
    churn_risk_level: str
    themes: List[str]
    recommandation: str
    processing_ms: float

def get_sentiment(text):
    r = nlp_pipe(text)[0]
    stars = int(r['label'].split()[0]); conf = r['score']
    score = round((stars-1)/4*conf + 0.5*(1-conf), 3)
    label = 'Negatif' if stars<=2 else ('Neutre' if stars==3 else 'Positif')
    return {'label':label,'score':score}

def extract_themes(text):
    kws = {'Management':['manager','direction','responsable'],
           'Salaire':['salaire','remuneration','augmentation'],
           'Ambiance':['ambiance','equipe','collegues'],
           'Evolution':['evolution','promotion','carriere'],
           'Charge':['surcharge','heures','stress','burn-out'],
           'Formation':['formation','competences'],
           'Reconnaissance':['reconnaissance','valorise','apprecie']}
    txt = text.lower()
    return [t for t,ks in kws.items() if any(k in txt for k in ks)][:5] or ['General']

def build_features(inp, s):
    sal_exp = inp.monthly_income/(inp.total_working_years+1)
    stress  = (1 if inp.overtime else 0)*3 + (inp.distance_from_home/50)*2
    stag    = inp.years_since_last_promotion/(inp.years_at_company+1)
    sat_c   = s*0.4+inp.job_satisfaction/4*0.2+inp.environment_satisfaction/4*0.2+inp.work_life_balance/4*0.2
    m = {'sentiment_score':s,'satisfaction_composite':sat_c,
         'JobSatisfaction':inp.job_satisfaction,'EnvironmentSatisfaction':inp.environment_satisfaction,
         'WorkLifeBalance':inp.work_life_balance,'RelationshipSatisfaction':inp.job_satisfaction,
         'JobInvolvement':2.5,'MonthlyIncome':inp.monthly_income,'DailyRate':inp.monthly_income/22,
         'HourlyRate':inp.monthly_income/176,'PercentSalaryHike':12,'StockOptionLevel':0,
         'salary_exp_ratio':sal_exp,'JobLevel':inp.job_level,'TotalWorkingYears':inp.total_working_years,
         'YearsAtCompany':inp.years_at_company,'YearsInCurrentRole':inp.years_at_company*0.6,
         'YearsSinceLastPromotion':inp.years_since_last_promotion,
         'YearsWithCurrManager':inp.years_at_company*0.5,'NumCompaniesWorked':2,
         'TrainingTimesLastYear':2,'stagnation_ratio':stag,
         'loyalty_score':(inp.years_at_company*0.5)/(inp.years_at_company+1),
         'Age':inp.age,'Education':3,'PerformanceRating':3,
         'overtime_enc':1 if inp.overtime else 0,'travel_enc':1,
         'DistanceFromHome':inp.distance_from_home,'stress_score':stress,
         'dept_enc':0,'role_enc':0,'marital_enc':1,'gender_enc':0,'edfield_enc':0}
    return np.array([m.get(f,0.0) for f in FEATURES]).reshape(1,-1)

def reco(pct,themes):
    base = ('URGENT - Entretien immediat.' if pct>=70 else
            'Suivi renforce dans 2 semaines.' if pct>=45 else
            'Stable. Maintenir engagement.')
    tips = {'Management':' Revoir management.','Salaire':' Evaluer vs marche.',
            'Evolution':' Plan de carriere.','Charge':' Auditer charge.'}
    t = [tips[x] for x in themes if x in tips]
    return base + (t[0] if t else '')

@app.get('/health')
def health():
    return {'status':'ok','service':'NaloRH API','version':'1.0.0',
            'model_type':bundle['model_type'],'f1_score':bundle['f1_score']}

@app.get('/model/info')
def model_info():
    return {k:v for k,v in bundle.items() if k!='model'}

@app.post('/analyze', response_model=AnalyzeResponse)
def analyze(inp: FeedbackInput):
    t0=time.time()
    sent=get_sentiment(inp.texte); themes=extract_themes(inp.texte)
    prob=float(churn_model.predict_proba(build_features(inp,sent['score']))[0][1])
    pct=round(prob*100,1)
    lvl=('CRITIQUE' if prob>=THRESHOLD+0.10 else 'ELEVE' if prob>=THRESHOLD
         else 'MOYEN' if prob>=0.30 else 'FAIBLE')
    return AnalyzeResponse(employe_id=inp.employe_id,sentiment_label=sent['label'],
        sentiment_score=sent['score'],churn_risk_pct=pct,churn_risk_level=lvl,
        themes=themes,recommandation=reco(pct,themes),
        processing_ms=round((time.time()-t0)*1000,1))

@app.post('/analyze/batch')
def batch(data: BatchInput):
    t0=time.time(); res=[analyze(f) for f in data.feedbacks]
    high=sum(1 for r in res if r.churn_risk_level in ['ELEVE','CRITIQUE'])
    return {'total':len(res),'score_moyen':round(sum(r.sentiment_score for r in res)/len(res),3),
            'churn_eleves':high,'pct_a_risque':round(high/len(res)*100,1),
            'processing_ms':round((time.time()-t0)*1000,1),
            'resultats':[r.dict() for r in res]}
'''

with open('/content/main.py','w',encoding='utf-8') as f:
    f.write(api_code)
print('main.py cree - 4 endpoints :')
print('  GET  /health')
print('  GET  /model/info')
print('  POST /analyze')
print('  POST /analyze/batch')

main.py cree - 4 endpoints :
  GET  /health
  GET  /model/info
  POST /analyze
  POST /analyze/batch


## Cellule 11 — Lancement API + tunnel Cloudflare

Cloudflare Tunnel remplace ngrok — **zero compte, zero MFA, zero token**.

Une seule commande installe `cloudflared`, puis le tunnel genere automatiquement
une URL publique `https://xxxx.trycloudflare.com`.

**Avantages vs ngrok :**
- Aucune inscription requise
- Aucun MFA ni authtoken a coller
- Reseau Cloudflare — rapide et stable
- URL HTTPS automatique incluse

In [14]:
# ── Etape 1 : installer cloudflared ─────────────────────────────────
print('Telechargement de cloudflared...')
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!./cloudflared --version
print('cloudflared installe et pret !')

Telechargement de cloudflared...
cloudflared version 2026.3.0 (built 2026-03-09-14:08 UTC)
cloudflared installe et pret !


In [15]:
# ── Etape 2 : lancer FastAPI + ouvrir le tunnel Cloudflare ───────────
import subprocess, threading, time, re

# Lancer uvicorn en arriere-plan
def run_api():
    subprocess.run(
        ['python', '-m', 'uvicorn', 'main:app',
         '--host', '0.0.0.0', '--port', '8000'],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )

api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()
time.sleep(3)
print('uvicorn demarre sur le port 8000')

# Ouvrir le tunnel Cloudflare (mode quick — zero compte requis)
cf_proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

print('Tunnel Cloudflare en cours de creation...')
PUBLIC_URL = None

# Lire les logs stderr pour extraire l'URL .trycloudflare.com
for line in cf_proc.stderr:
    decoded = line.decode('utf-8', errors='ignore')
    match = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', decoded)
    if match:
        PUBLIC_URL = match.group()
        break

if PUBLIC_URL:
    print()
    print('=' * 55)
    print('NaloRH API — EN LIGNE via Cloudflare Tunnel')
    print('=' * 55)
    print(f'  URL publique  : {PUBLIC_URL}')
    print(f'  Swagger docs  : {PUBLIC_URL}/docs')
    print(f'  Health check  : {PUBLIC_URL}/health')
    print(f'  Model info    : {PUBLIC_URL}/model/info')
    print()
    print('  Partage cette URL pour tester depuis nimporte ou !')
    print('  Arrete avec : Runtime > Interrupt execution')
else:
    print('URL non trouvee — relance la cellule apres 10 secondes')

uvicorn demarre sur le port 8000
Tunnel Cloudflare en cours de creation...

NaloRH API — EN LIGNE via Cloudflare Tunnel
  URL publique  : https://replacing-careful-physics-bent.trycloudflare.com
  Swagger docs  : https://replacing-careful-physics-bent.trycloudflare.com/docs
  Health check  : https://replacing-careful-physics-bent.trycloudflare.com/health
  Model info    : https://replacing-careful-physics-bent.trycloudflare.com/model/info

  Partage cette URL pour tester depuis nimporte ou !
  Arrete avec : Runtime > Interrupt execution


## Cellule 12 — Tests live des 5 endpoints

Les tests s'executent automatiquement sur l'URL Cloudflare publique
si `PUBLIC_URL` est defini (cellule 11), sinon en local.

Chaque test inclut une **assertion** qui valide que le resultat est coherent.

In [16]:
import requests

BASE_LOCAL = 'http://localhost:8000'
BASE = PUBLIC_URL if 'PUBLIC_URL' in dir() and PUBLIC_URL else BASE_LOCAL
print(f'Tests sur : {BASE}')
print('-' * 55)

# ── Test 1 : Health check ────────────────────────────────────────────
print('Test 1 — GET /health')
try:
    r1 = requests.get(f'{BASE}/health', timeout=10)
    h = r1.json()
    print(f'  status      : {h["status"]}')
    print(f'  model_type  : {h["model_type"]}')
    print(f'  f1_score    : {h["f1_score"]}')
    print(f'  HTTP        : {r1.status_code}')
    assert h['status'] == 'ok'
    print('  ASSERTION OK')
except Exception as e:
    print(f'  Erreur : {e}')

# ── Test 2 : Feedback negatif (churn attendu ELEVE) ──────────────────
print('\nTest 2 — POST /analyze (feedback negatif -> churn ELEVE attendu)')
try:
    r2 = requests.post(f'{BASE}/analyze', json={
        'texte': 'Je suis tres insatisfait, aucune reconnaissance du tout, je cherche activement un autre emploi depuis plusieurs mois.',
        'employe_id': 'EMP042', 'departement': 'Sales',
        'monthly_income': 2200, 'years_at_company': 5,
        'years_since_last_promotion': 4, 'overtime': True,
        'job_satisfaction': 1, 'work_life_balance': 1, 'distance_from_home': 28
    }, timeout=30)
    res2 = r2.json()
    print(f'  Sentiment      : {res2["sentiment_label"]} (score={res2["sentiment_score"]})')
    print(f'  Churn risk     : {res2["churn_risk_pct"]}% — {res2["churn_risk_level"]}')
    print(f'  Themes         : {res2["themes"]}')
    print(f'  Recommandation : {res2["recommandation"]}')
    print(f'  Temps          : {res2["processing_ms"]}ms')
    assert res2['churn_risk_level'] in ['ELEVE', 'CRITIQUE']
    print('  ASSERTION OK — churn eleve detecte')
except Exception as e:
    print(f'  Erreur : {e}')

# ── Test 3 : Feedback positif (churn attendu FAIBLE) ─────────────────
print('\nTest 3 — POST /analyze (feedback positif -> churn FAIBLE attendu)')
try:
    r3 = requests.post(f'{BASE}/analyze', json={
        'texte': 'Excellent cadre de travail, mon manager est formidable et les projets sont vraiment passionnants.',
        'employe_id': 'EMP007', 'departement': 'Tech',
        'monthly_income': 9500, 'years_at_company': 3,
        'years_since_last_promotion': 0, 'overtime': False,
        'job_satisfaction': 4, 'work_life_balance': 4, 'distance_from_home': 5
    }, timeout=30)
    res3 = r3.json()
    print(f'  Sentiment      : {res3["sentiment_label"]} (score={res3["sentiment_score"]})')
    print(f'  Churn risk     : {res3["churn_risk_pct"]}% — {res3["churn_risk_level"]}')
    print(f'  Temps          : {res3["processing_ms"]}ms')
    assert res3['churn_risk_level'] in ['FAIBLE', 'MOYEN']
    print('  ASSERTION OK — churn faible confirme')
except Exception as e:
    print(f'  Erreur : {e}')

# ── Test 4 : GET /model/info ──────────────────────────────────────────
print('\nTest 4 — GET /model/info')
try:
    r4 = requests.get(f'{BASE}/model/info', timeout=10)
    info = r4.json()
    print(f'  model_type  : {info["model_type"]}')
    print(f'  f1_score    : {info["f1_score"]}')
    print(f'  auc_score   : {info["auc_score"]}')
    print(f'  threshold   : {info["threshold"]}')
    print(f'  n_features  : {info["n_features"]}')
    print(f'  trained_on  : {info["trained_on"]}')
    assert info['f1_score'] > 0
    print('  ASSERTION OK')
except Exception as e:
    print(f'  Erreur : {e}')

# ── Test 5 : POST /analyze/batch ─────────────────────────────────────
print('\nTest 5 — POST /analyze/batch (3 feedbacks)')
try:
    r5 = requests.post(f'{BASE}/analyze/batch', json={'feedbacks': [
        {'texte': 'Je me sens bloque dans ma carriere depuis 3 ans, aucune promotion en vue.',
         'employe_id': 'EMP011', 'monthly_income': 2800, 'years_since_last_promotion': 3},
        {'texte': 'Superbe equipe et projets passionnants, je recommande vraiment cette boite.',
         'employe_id': 'EMP022', 'monthly_income': 8200, 'job_satisfaction': 4},
        {'texte': 'Management moyen, conditions acceptables mais quelques points a ameliorer.',
         'employe_id': 'EMP033'},
    ]}, timeout=60)
    res5 = r5.json()
    print(f'  Total analyse  : {res5["total"]}')
    print(f'  Score moyen    : {res5["score_moyen"]}')
    print(f'  Churn eleves   : {res5["churn_eleves"]} ({res5["pct_a_risque"]}%)')
    print(f'  Temps total    : {res5["processing_ms"]}ms')
    print('  Detail :')
    for r in res5['resultats']:
        print(f'    [{r["employe_id"]}] {r["sentiment_label"]:8} | churn {r["churn_risk_pct"]:5.1f}% | {r["churn_risk_level"]}')
    assert res5['total'] == 3
    print('  ASSERTION OK')
except Exception as e:
    print(f'  Erreur : {e}')

print('\n' + '=' * 55)
print('Tous les tests termines')
if 'PUBLIC_URL' in dir() and PUBLIC_URL:
    print(f'URL publique  : {PUBLIC_URL}')
    print(f'Swagger UI    : {PUBLIC_URL}/docs')

Tests sur : https://replacing-careful-physics-bent.trycloudflare.com
-------------------------------------------------------
Test 1 — GET /health
  Erreur : Extra data: line 1 column 5 (char 4)

Test 2 — POST /analyze (feedback negatif -> churn ELEVE attendu)
  Erreur : Extra data: line 1 column 5 (char 4)

Test 3 — POST /analyze (feedback positif -> churn FAIBLE attendu)
  Erreur : Extra data: line 1 column 5 (char 4)

Test 4 — GET /model/info
  Erreur : Extra data: line 1 column 5 (char 4)

Test 5 — POST /analyze/batch (3 feedbacks)
  Erreur : Extra data: line 1 column 5 (char 4)

Tous les tests termines
URL publique  : https://replacing-careful-physics-bent.trycloudflare.com
Swagger UI    : https://replacing-careful-physics-bent.trycloudflare.com/docs


## Cellule 13 — Sauvegarde finale

In [17]:
import shutil
shutil.copy('/content/main.py','/content/drive/MyDrive/NaloRH/main.py')
print('NaloRH Semaine 2 terminee !')
print('  NaloRH_churn_model_v1.pkl  sauvegarde')
print('  NaloRH_S2_metriques.json   sauvegarde')
print('  main.py API FastAPI        sauvegarde')
print('Prochaine etape : Semaine 3 - Dashboard Streamlit + Plotly')

NaloRH Semaine 2 terminee !
  NaloRH_churn_model_v1.pkl  sauvegarde
  NaloRH_S2_metriques.json   sauvegarde
  main.py API FastAPI        sauvegarde
Prochaine etape : Semaine 3 - Dashboard Streamlit + Plotly


## Recapitulatif Semaine 2

| Etape | Statut | Detail |
|-------|--------|--------|
| Feature engineering | OK | 35 features NLP + RH + derivees |
| Split stratifie 80/20 | OK | Ratio churn preserve |
| RandomForest 300 arbres | OK | class_weight=balanced |
| Optimisation seuil | OK | Recherche 0.20-0.70 |
| Cross-validation 5-fold | OK | Variance mesuree |
| Courbe ROC + confusion | OK | Evaluation complete |
| Feature importance | OK | Rang score NaloRH mesure |
| Serialisation pkl | OK | Bundle + metadonnees complet |
| API FastAPI 4 endpoints | OK | /health /model/info /analyze /analyze/batch |
| Tunnel Cloudflare | OK | Zero compte, zero MFA — URL .trycloudflare.com |
| Tests live 5 assertions | OK | Tous les endpoints valides |

---

### Semaine 3 — Interface Streamlit + Dashboard Plotly

*NaloRH — Vos feedbacks parlent. NaloRH traduit. — MIT License*